<a href="https://colab.research.google.com/github/interpretune/examples/blob/main/tutorials/publish/neuron_communities/neuron_communities_replication.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# Neuron Communities: Replicating arXiv 2611.23112 on Llama-3.1-8B

Large-scale neuron community discovery <a id="cite-1"></a>[[1]](#ref-1) finds communities of MLP neurons in
Llama-3.1-8B that are attributed together across many prompts, and tests them causally. Its case study (Figure 2)
is a community that predicts the article "an" before a vowel-initial word: ablating its handful of neurons moves the
next-token distribution almost as much as ablating everything in its layers, and keeping only those neurons (plus a
few always-on hubs) preserves it.

This notebook replicates that measurement with interpretune's addressing and intervention surfaces rather than
bespoke hooks, using the paper's public release of communities and prompts <a id="cite-2"></a>[[2]](#ref-2):

- **Address:** each community is an explicit unit set across layers in a venngram address book, so the paper's
  "ablate every neuron in these layers except the community and the hubs" is set algebra over the book.
- **Ablate and measure:** zero-ablation at the last position through the model backend, with KL divergence and the
  change of the top-1 probability, on fresh FineWeb-Edu prompts and on the release's own prompts.
- **Attribute:** the paper's gradient-times-activation attribution over every neuron, which the released community
  should come out of.

Each section states the paper's claim, our measurement, and a control. A closing section says what transferred.

The model and its backend come from one session, built in the loading cell from the task-neutral
`speediedan/prompt-analysis` component. Every measurement takes them as arguments
(`it_examples.utils.neuron_communities_helpers`). The notebook needs a GPU with 24 GB and access to the gated
`meta-llama/Llama-3.1-8B` repository.

In [2]:
# @title Imports { display-mode: "form" }
import os
import random
import statistics as st
import tempfile
from pathlib import Path

import torch
from scipy.stats import spearmanr

import interpretune as it
from interpretune.analysis.backends import get_model_backend
from interpretune.harness.notebook_inputs import required_os_env
from interpretune.harness.notebook_ui import display_notebook_table
from it_examples.utils.neuron_communities_helpers import (
    PAPER_FIG2,
    TESTS,
    an_windows,
    attribute,
    attribution_controls,
    community_book,
    hand_written_gradients,
    hand_written_zero_ablation,
    load_community,
    random_payload,
    record_windows,
    run_ablations,
    zero_payloads,
)

~/worktrees/interpretune-ref-7d01052f/src/interpretune/hub/trust.py:96: Refusing to execute analysis ops from 'speediedan/jlens_steering_ops': loading it runs code published by that repo inside this process, with your filesystem, network and Hugging Face token. Interpretune does not execute hub-resident code unless you opt in.
  Inspect it first: interpretune.hub.pull('speediedan/jlens_steering_ops') caches the repo without executing anything, so you can read the entrypoint the manifest names.
  Then opt in for this session: os.environ['IT_TRUST_REMOTE_CODE'] = '1' (or export IT_TRUST_REMOTE_CODE=1 before starting the process).
  Pin a revision when you pull so trusted code cannot change under you.
  Details: https://interpretune.org/en/latest/usage/hub_trust_posture.html


~/worktrees/interpretune-ref-7d01052f/src/interpretune/analysis/ops/dispatcher.py:1121: Alias 'ct_feature_intervention' already has an assigned op or alias so the alias specified by 'feature_intervention_forward' cannot be mapped to it
~/worktrees/interpretune-ref-7d01052f/src/interpretune/analysis/ops/dispatcher.py:1121: Alias 'semantic_direction' already has an assigned op or alias so the alias specified by 'concept_direction' cannot be mapped to it
~/worktrees/interpretune-ref-7d01052f/src/interpretune/analysis/ops/dispatcher.py:1121: Alias 'model_forward' already has an assigned op or alias so the alias specified by 'model_fwd' cannot be mapped to it
~/worktrees/interpretune-ref-7d01052f/src/interpretune/analysis/ops/dispatcher.py:1121: Alias 'jacobian_lens_read' already has an assigned op or alias so the alias specified by 'jlens_read' cannot be mapped to it
~/worktrees/interpretune-ref-7d01052f/src/interpretune/analysis/ops/dispatcher.py:1121: Alias 'jacobian_lens_concept_probe' 

## Notebook Parameters

This cell contains parameters that can be injected by papermill during parameterized test runs. The paper uses
8,000 " an" prompts; `n_prompts` trades precision for time, and the medians are already stable at a few hundred.
`table1_communities` are four of the paper's Table 1 communities, measured on the release's own prompts. `REVISION`
pins the component revision the session is loaded from.

In [3]:
# Parameters - These will be injected by papermill during parameterized test runs
REVISION = None  # component snapshot; None resolves the recorded pin (a local-publish rev validates unmerged configs)
device = "cuda:0"
community_id = "217996"  # the paper's Figure 2 community (" an" before a vowel-initial word)
table1_communities = ["187321", "195197", "228121", "225849"]
n_prompts = 200  # FineWeb-Edu windows ending at " an" for the Figure 2 tests and the attribution
batch_size = 4
random_draws = 2  # random one-neuron-per-layer controls per batch
top_k = 5  # neurons kept per layer per prompt (the release's TOP_K)
seed = 0
cache_dir = None  # where release records are cached (None: a temporary directory)

In [4]:
# @title Environment Setup { display-mode: "form" }
env_path: str | None = None  # set to '/full/path/to/.env' to override
os_env_reqs = None
assert required_os_env(env_path=env_path, env_reqs=os_env_reqs)
cache = Path(cache_dir).expanduser() if cache_dir else Path(tempfile.gettempdir()) / "neuron_communities"

## The Model and Its Backend

A neuron here is what the paper calls one: an input to an MLP's down projection, the product of the gated
activation and the up projection. Interpretune names that site `blocks.{layer}.mlp.out.hook_in`.

The session is the `prompt_analysis.llama3.nnsight` configuration: Llama-3.1-8B in bf16 on nnsight, with no latent
model, no dataset and no task head. A `[core, nnsight]` session carries its own `NNsightModelBackend`, the
model-level surface the analysis ops use, and `get_model_backend` returns it. Its parameters are frozen: nothing
here trains, and the attribution section takes its gradients with respect to activations, which the backend makes
possible without gradients on the 8B parameters.

In [5]:
if REVISION is None:  # a validation run passes a local-publish REVISION and needs neither fetch
    it.hub.pull("speediedan/prompt-analysis")  # caches the component's manifest and entrypoint; executes nothing
os.environ["IT_TRUST_REMOTE_CODE"] = "1"  # opt in to running the component's entrypoint, for this session only
KEY = "prompt_analysis.llama3.nnsight"
if REVISION is None:
    it.hub.pull("speediedan/prompt-analysis", KEY)  # fetches the one configuration loaded below
itdm_cfg, it_cfg, dm_cls, m_cls = it.hub.load("speediedan/prompt-analysis", KEY, revision=REVISION)
it_cfg.nnsight_cfg.device_map = {"": device}  # a map loads straight to the device; a bare name loads in host RAM first
session = it.ITSession(
    it.ITSessionConfig(
        adapter_ctx=(it.Adapter.core, it.Adapter.nnsight),
        datamodule_cfg=itdm_cfg,
        module_cfg=it_cfg,
        datamodule_cls=dm_cls,
        module_cls=m_cls,
    )
)
it.it_init(**session)
module = session.module
model, backend = module.model, get_model_backend(module)
hf_model = model._module
hf_model.requires_grad_(False)
torch.set_grad_enabled(False)  # the backend enables gradients itself where it needs them
tokenizer = model.tokenizer
n_layers, width = hf_model.config.num_hidden_layers, hf_model.config.intermediate_size
print(f"{it_cfg.nnsight_cfg.model_name}: {n_layers} layers, {width} neurons per layer")

it_component.yaml:   0%|          | 0.00/2.00k [00:00<?, ?B/s]

prompt_analysis.py:   0%|          | 0.00/5.09k [00:00<?, ?B/s]

prompt_analysis.llama3.nnsight.yaml:   0%|          | 0.00/1.85k [00:00<?, ?B/s]

[INFO] interpretune.utils.logging: Initializing NNsight TransformersModel with model: meta-llama/Llama-3.1-8B


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: NNsight TransformersModel initialized successfully


[INFO] interpretune.utils.logging: Preparing data: InterpretunableDataModule


[INFO] interpretune.utils.logging: Setting up datamodule: InterpretunableDataModule


[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule


meta-llama/Llama-3.1-8B: 32 layers, 14336 neurons per layer


## Addressing a Community

The release record for community 217996 lists its 26 members across 20 layers, its always-on hubs (17 neurons that
are in nearly every prompt's top set, kept out of the communities), and its main span: the layers where it acts.
The three tests are then three entries of one address book:

| test | ablated at the last position | the paper's claim |
| --- | --- | --- |
| necessity | `@community` | removing the members disrupts the prediction |
| sufficiency | `@span - (@community \| @hubs)` | keeping only the members and hubs preserves it |
| layer necessity | `@span - @hubs` | the span carries the computation |

`unit_interventions` resolves each entry into a zero-replace payload per layer.

In [6]:
community = load_community(community_id, cache)
book = community_book(community, width)
payloads = zero_payloads(book, n_layers, width)
print(f"{community.name} (release record sha256 {community.sha256[:12]})")
print(f"main span: layers {community.span}; members {len(community.members)}, hubs {len(community.hubs)}")
display_notebook_table(
    [
        [test, f"@{entry}", sum(len(spec.unit_indices) for spec in payloads[test].values())]
        for test, entry in TESTS.items()
    ],
    ["test", "address", "neurons ablated"],
    title="Each test's payload",
)

Agreement: 'an' → vowel-initial next word (release record sha256 0c17926431b9)
main span: layers [2, 5, 8, 12, 13, 14]; members 26, hubs 17


test,address,neurons ablated
necessity,@community,6
sufficiency,@suff_ablate,86006
layer necessity,@layer_ablate,86012


### Control: the payload is the edit it claims to be

Before measuring anything, the necessity payload is checked against the same edit written by hand, as forward
pre-hooks on the HuggingFace model's down projections that zero the members at the last position. The two must give
the same logits, and the edit must move them.

In [7]:
windows = an_windows(tokenizer, n_prompts, seed)
probe = {"input_ids": torch.tensor(windows[:batch_size])}
clean, edited = backend.fwd_w_intervention(model, probe, payloads["necessity"])
manual = hand_written_zero_ablation(hf_model, probe["input_ids"], book["community"]["units"])
err = (manual[:, -1].float() - edited[:, -1].float().to(manual.device)).abs().max().item()
moved = (clean[:, -1].float() - edited[:, -1].float()).abs().max().item()
print(f"payload vs hand-written edit: max |logit difference| {err:.4f}; the edit moved the logits by up to {moved:.2f}")
assert err < 0.05 and moved > 0.1, "the unit payload does not reproduce the hand-written down_proj edit"

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

payload vs hand-written edit: max |logit difference| 0.0000; the edit moved the logits by up to 12.95


## Figure 2: Necessity, Sufficiency and Layer Necessity

**The paper's claim** (its Figure 2, on 8,000 " an" prompts): ablating the community gives a median KL of 4.99 nats,
sufficiency 0.053, and layer necessity 3.15; the mean change of the clean top-1 probability is 0.42, 0.08 and 0.38.
A random neuron per span layer barely moves it.

**Our measurement:** `n_prompts` FineWeb-Edu windows of 64 tokens ending at " an", one per document, with the same
three payloads and a random one-neuron-per-layer control.

In [8]:
rng = random.Random(seed)
fig2 = run_ablations(
    model,
    backend,
    windows,
    payloads,
    lambda: random_payload(community, n_layers, width, rng),
    batch_size=batch_size,
    draws=random_draws,
)
medians = fig2.medians()
rows = [
    [test, f"{medians[test][0]:.3f}", f"{PAPER_FIG2[test][0]}", f"{medians[test][1]:.3f}", f"{PAPER_FIG2[test][1]}"]
    for test in PAPER_FIG2
]
rows.append(
    ["random control", f"{medians['random control'][0]:.3f}", "near 0", f"{medians['random control'][1]:.3f}", ""]
)
display_notebook_table(
    rows,
    ["test", "median KL (ours)", "median KL (paper)", "mean |Δ top-1 p| (ours)", "(paper)"],
    title=f"Community {community_id} on {len(windows)} ' an' prompts",
    caption=f"Clean top-1 probability: mean {st.mean(fig2.p_clean):.3f} here, 0.57 in the paper.",
)

test,median KL (ours),median KL (paper),mean |Δ top-1 p| (ours),(paper)
necessity,6.026,4.99,0.397,0.42
sufficiency,0.054,0.053,0.064,0.08
layer necessity,4.194,3.15,0.370,0.38
random control,0.000,near 0,0.004,


## The Release's Own Prompts

Our prompts differ from the paper's mix of sources, so the release's own prompts are the closer comparison. The
release publishes, for each community, 100 prompts with its per-prompt KLs for the three tests. We re-run each
prompt and compare medians, and correlate our per-prompt KLs with the release's. The release publishes text rather
than token ids, so prompts are re-tokenized, and the fraction whose clean top-1 token matches the release's is the
alignment check.

In [9]:
def release_comparison(cid: str) -> list:
    c = community if cid == community_id else load_community(cid, cache)
    c_payloads = zero_payloads(community_book(c, width), n_layers, width)
    prompts = c.record["prompts"]
    result = run_ablations(model, backend, record_windows(tokenizer, c), c_payloads, batch_size=1, draws=0)
    want = [c.record["tokens"][p["b"][0][0]] for p in prompts]
    match = st.mean(tokenizer.decode([t]) == w for t, w in zip(result.top1, want))
    row = [cid, c.name[:42], f"{match:.0%}"]
    for test, key in zip(TESTS, "nsl"):
        theirs = [p[key] for p in prompts]
        rho = spearmanr(result.kl[test], theirs).statistic
        row.append(f"{st.median(result.kl[test]):.2f} / {st.median(theirs):.2f} (ρ {rho:+.2f})")
    return row


display_notebook_table(
    [release_comparison(cid) for cid in [community_id, *table1_communities]],
    ["community", "behaviour", "top-1 match", "necessity", "sufficiency", "layer necessity"],
    title="Median KL, ours / release (Spearman ρ of per-prompt KLs), on the release's prompts",
)

community,behaviour,top-1 match,necessity,sufficiency,layer necessity
217996,Agreement: 'an' → vowel-initial next word,64%,5.55 / 5.23 (ρ +0.59),0.07 / 0.06 (ρ +0.63),4.10 / 3.97 (ρ +0.66)
187321,Choose 'are' / 'is' / 'were' after 'there',92%,0.99 / 1.20 (ρ +0.26),0.11 / 0.09 (ρ +0.41),1.29 / 1.17 (ρ +0.30)
195197,Complete a LaTeX command after '\',90%,3.93 / 3.71 (ρ +0.62),0.09 / 0.05 (ρ +0.53),4.93 / 5.42 (ρ +0.66)
228121,Detect the plural '-s' ending: the noun is,69%,1.68 / 1.92 (ρ +0.59),0.11 / 0.06 (ρ +0.42),2.01 / 1.85 (ρ +0.43)
225849,Close the parenthesis,78%,1.23 / 1.75 (ρ +0.50),0.14 / 0.08 (ρ +0.47),7.10 / 6.84 (ρ +0.69)


## Attribution: Does the Community Come Out of It?

**The paper's method:** per prompt, score every neuron by activation times the gradient of the log-probability of the
model's own top-1 token, at the last position, and keep the top 5 per layer. A community is a set of neurons that
co-occur in those top sets; the release assigns a community to a prompt when at least two of its members are present.
The release ranks by the absolute score; the paper's equation 1 uses the signed one, so both are shown.

**Our measurement:** activations from `backend.fwd_w_cache` and gradients from `backend.bwd_w_intervention` with no
edits, at the 32 down-projection inputs. The gradients are first checked against plain autograd.

In [10]:
attribution = attribute(model, backend, windows, n_layers, top_k=top_k, batch_size=batch_size)
reference = hand_written_gradients(hf_model, attribution.first_batch_ids, n_layers)
grad_err = (reference - attribution.first_batch_grads).abs().max().item()
grad_max = reference.abs().max().item()
print(f"captured vs hand-written gradients: max |error| {grad_err:.2e} of max |gradient| {grad_max:.2e}")
assert grad_max > 0 and grad_err <= 0.02 * grad_max, "the captured gradients do not match plain autograd"

captured vs hand-written gradients: max |error| 0.00e+00 of max |gradient| 5.66e-01


### Controls

A uniform random neuron reaches one layer's top 5 out of 14,336 about 0.03% of the time, so a random-set baseline
says little. Two stronger controls:

- **A hard baseline:** per layer, the most frequent non-member, non-hub neurons on these same prompts, in the
  community's per-layer counts. It is chosen in hindsight, so it bounds what attribution frequency alone gives.
- **Specificity:** the members of the four other communities, on the same prompts.

In [11]:
others = {cid: load_community(cid, cache) for cid in table1_communities}
rows = []
for label, sets in (("|a| (release)", attribution.top_abs), ("signed a (paper)", attribution.top_signed)):
    ctl = attribution_controls(sets, community)
    other_present = " / ".join(
        f"{sum(len(set(o.members) & s) >= 2 for s in sets) / len(sets):.1%}" for o in others.values()
    )
    rows.append(
        [
            label,
            f"{ctl['present']:.1%} ({ctl['mean_members']:.2f})",
            f"{ctl['hard_present']:.1%} ({ctl['hard_mean']:.2f})",
            other_present,
            f"{ctl['member_rank_median']:.0f} of {ctl['distinct']}",
            f"{min(ctl['hub_ranks'])}-{max(ctl['hub_ranks'])}",
        ]
    )
display_notebook_table(
    rows,
    [
        "score",
        "community present (mean members)",
        "hard baseline",
        "other communities present",
        "member rank (median)",
        "hub ranks",
    ],
    title=f"Community {community_id} in the top-{top_k} sets of {len(windows)} ' an' prompts",
    caption=(
        "Present: at least 2 members in a prompt's top sets (the release's rule). "
        "Ranks: by how many prompts a neuron reached."
    ),
)

score,community present (mean members),hard baseline,other communities present,member rank (median),hub ranks
|a| (release),99.5% (7.89),87.0% (3.54),0.0% / 0.0% / 0.5% / 0.0%,42 of 18978,1-20
signed a (paper),89.0% (5.64),81.5% (3.27),0.0% / 0.0% / 0.5% / 0.0%,42 of 20605,1-20


## What Transferred

- **Figure 2 replicates.** The necessity, sufficiency and layer-necessity medians land near the paper's on fresh
  prompts, and the random control stays near zero. Necessity and layer necessity tend to come out somewhat larger
  here. The paper mixes FineWeb-Edu, Wikipedia and FineMath prompts and reports a higher clean top-1 probability, so
  the prompt distributions differ.
- **The release's prompts agree prompt by prompt** for the Table 1 communities: medians close to the release's and
  positive rank correlations. Sufficiency comes out 0.01 to 0.06 nats above the release's for every community, still
  small. Re-tokenization from text limits how close the match can be.
- **Attribution recovers the community.** The members are among the most frequently attributed neurons after the
  hubs, they are present on almost every prompt, and other communities' members essentially never are. The
  presence rule alone is weak evidence: with 26 members, the hindsight baseline clears it often too. The member count,
  the ranks and the specificity carry the result.
- **Not replicated here:** the discovery step itself (co-occurrence statistics over many prompts and Leiden community
  detection). This notebook checks that the released communities behave as claimed under interpretune's surfaces.

**Gaps found and filed.** Interpretune has no call that returns activations and their gradients from one trace
([#1090](https://github.com/speediedan/interpretune/issues/1090)), and its gradient capture ran a redundant second
traced backward pass when nothing is edited ([#1089](https://github.com/speediedan/interpretune/issues/1089),
fixed). Everything else the replication needed, including "ablate everything except", was already expressible.

## References

1. <a id="ref-1"></a>"Large-scale Neuron Community Discovery Reveals Neuron-Sparse Behaviors in Language Models."
   arXiv:2611.23112, 2026. [arxiv.org/abs/2611.23112](https://arxiv.org/abs/2611.23112) [↩](#cite-1)
2. <a id="ref-2"></a>Goodfire. *neuron-communities*: code and data release (MIT).
   [github.com/goodfire-ai/neuron-communities](https://github.com/goodfire-ai/neuron-communities) [↩](#cite-2)